# **Introducción al análisis de datos en Python** 
#### Profesor: Santiago Neira

## Clase 5. Visualización de datos

### Objetivos de la clase

1. Entender **qué tipo de gráfico** usar según los datos y la pregunta que queremos responder
2. Crear visualizaciones con **Matplotlib** (a través de Pandas)
3. Crear visualizaciones con **Seaborn** para análisis más avanzados
4. Personalizar gráficos para comunicar información de forma efectiva

---
## Bloque 1: ¿Qué gráfico uso?

Antes de escribir una sola línea de código, la pregunta más importante es: **¿Qué quiero comunicar?**

El tipo de gráfico depende de:
- El **tipo de variable** (numérica vs. categórica)
- La **cantidad de variables** que queremos visualizar
- La **pregunta** que queremos responder

### Guía visual: tipos de gráficos

La siguiente imagen es el **Visual Vocabulary** del Financial Times. Organiza los gráficos por el tipo de relación que quieres mostrar:

<img src="https://raw.githubusercontent.com/Financial-Times/chart-doctor/main/visual-vocabulary/poster.png" width="900">

*Fuente: [FT Visual Vocabulary](https://github.com/Financial-Times/chart-doctor/tree/main/visual-vocabulary) - Licencia abierta*

### Resumen práctico: ¿Qué gráfico para qué?

| Pregunta | Tipo de datos | Gráfico recomendado | `kind=` en Pandas |
|----------|--------------|---------------------|-------------------|
| ¿Cómo se **distribuye** una variable numérica? | 1 numérica | Histograma | `'hist'` |
| ¿Cómo se comparan las **distribuciones** entre grupos? | 1 numérica + 1 categórica | Boxplot | `'box'` |
| ¿Cuántos hay **por categoría**? | 1 categórica | Gráfico de barras | `'bar'` / `'barh'` |
| ¿Cómo **evoluciona** algo en el tiempo? | Temporal + numérica | Gráfico de línea | `'line'` |
| ¿Existe **relación** entre dos variables numéricas? | 2 numéricas | Gráfico de dispersión | `'scatter'` |
| ¿Qué **proporción** tiene cada parte del total? | 1 categórica | Gráfico de torta* | `'pie'` |

> **\*Nota sobre gráficos de torta:** En la práctica profesional, los gráficos de barras casi siempre comunican proporciones de manera más clara que los de torta. El ojo humano compara longitudes mejor que ángulos. Dicho esto, es importante saber construirlos y personalizarlos.

---
## Bloque 2: Matplotlib desde Pandas

La forma más rápida de graficar en Python es usar el método `.plot()` directamente sobre un DataFrame de Pandas. Internamente, Pandas usa **Matplotlib** para generar los gráficos.

### Configuración inicial

In [ ]:
# Instalación (solo la primera vez)
# %pip install matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Cargamos los datos

Vamos a trabajar con un dataset de **accidentes vehiculares en Bogotá durante 2016**. Incluye información sobre fecha, hora, localidad, gravedad, tipo de accidente, condiciones climáticas, número de heridos y muertos, entre otros.

In [ ]:
base_de_datos = pd.read_csv('./data/info_accidentes.csv')
base_de_datos['Fecha'] = pd.to_datetime(base_de_datos['Fecha'])
base_de_datos.head()

In [ ]:
# Veamos qué columnas tenemos disponibles
base_de_datos.columns.tolist()

In [ ]:
# Dimensiones del dataset
print(f'Filas: {base_de_datos.shape[0]:,}')
print(f'Columnas: {base_de_datos.shape[1]}')

Algunas columnas interesantes para visualizar:
- `GravedadNombre`: Solo Daños, Con Heridos, Con Muertos
- `ClaseNombre`: Choque, Atropello, Volcamiento, Caída de ocupante, etc.
- `TipoDiseño`: Intersección, Tramo de Vía, Glorieta, Puente, etc.
- `TipoTiempo`: Normal, Lluvia, Viento, Niebla
- `Localidad`: 19 localidades de Bogotá
- `TotalMuertos`, `TotalHeridos`: variables numéricas

---
### 2.1 Gráfico de línea: Evolución temporal

**Pregunta:** ¿Cómo evolucionó la cantidad de accidentes a lo largo del año?

In [ ]:
# Primero, preparamos los datos: agrupamos por fecha y contamos
accidentes_diarios = (base_de_datos
    .groupby('Fecha')
    .size()
    .reset_index(name='Número de accidentes'))

accidentes_diarios.head()

Ahora graficamos. Lo más básico:

In [ ]:
accidentes_diarios.plot(x='Fecha', y='Número de accidentes', kind='line');

Funciona, pero se ve pequeño y sin contexto. Vamos mejorándolo paso a paso:

In [ ]:
# Agregamos tamaño y título
accidentes_diarios.plot(
    x='Fecha', 
    y='Número de accidentes', 
    kind='line',
    figsize=(18, 4),
    title='Accidentes vehiculares diarios en Bogotá (2016)'
);

In [ ]:
# Cambiamos el color y quitamos la leyenda (solo hay una serie, no aporta)
accidentes_diarios.plot(
    x='Fecha', 
    y='Número de accidentes', 
    kind='line',
    figsize=(18, 4),
    title='Accidentes vehiculares diarios en Bogotá (2016)',
    color='darkred',
    legend=False
)
plt.ylabel('Cantidad')
plt.show()

**Parámetros clave de `.plot()`:**

| Parámetro | Qué hace | Ejemplo |
|-----------|----------|--------|
| `kind` | Tipo de gráfico | `'line'`, `'bar'`, `'barh'`, `'hist'`, `'scatter'`, `'box'`, `'pie'` |
| `figsize` | Tamaño `(ancho, alto)` en pulgadas | `(18, 4)` |
| `title` | Título del gráfico | `'Mi gráfico'` |
| `color` | Color de línea/barras/puntos | `'darkred'`, `'#e74c3c'` |
| `legend` | ¿Mostrar leyenda? | `True` / `False` |
| `xlabel`, `ylabel` | Etiquetas de los ejes | `'Fecha'` |
| `rot` | Rotación de etiquetas del eje x | `45` |

---
### 2.2 Histograma: Distribución de una variable numérica

**Pregunta:** ¿A qué horas del día ocurren más accidentes?

In [ ]:
# Primero extraemos la hora como número (0-23)
base_de_datos['Hora'] = pd.to_datetime(base_de_datos['HoraOcurrencia']).dt.hour
base_de_datos['Hora'].head(10)

In [ ]:
# Histograma básico
base_de_datos['Hora'].plot(kind='hist');

In [ ]:
# Mejoramos: más bins para ver cada hora, colores, bordes
base_de_datos['Hora'].plot(
    kind='hist', 
    bins=24,
    color='navy',
    edgecolor='white',
    figsize=(12, 4),
    title='Distribución de accidentes por hora del día'
)
plt.xlabel('Hora del día')
plt.ylabel('Frecuencia')
plt.show()

Hay dos picos de accidentalidad: alrededor de las **7-8 AM** y las **2-3 PM**, que coinciden con las horas pico de tráfico.

**Pregunta:** ¿Cómo se distribuyen los accidentes por mes?

In [ ]:
base_de_datos['Mes'] = base_de_datos['Fecha'].dt.month

base_de_datos['Mes'].plot(
    kind='hist',
    bins=12,
    color='teal',
    edgecolor='white',
    figsize=(10, 4),
    title='Distribución de accidentes por mes'
)
plt.xlabel('Mes')
plt.ylabel('Frecuencia')
plt.show()

---
### 2.3 Gráfico de barras: Conteo por categoría

**Pregunta:** ¿Cuántos accidentes ocurrieron en cada localidad?

In [ ]:
# Contamos accidentes por localidad
accidentes_por_localidad = base_de_datos['Localidad'].value_counts()
accidentes_por_localidad

In [ ]:
# Gráfico de barras horizontal (barh) - útil cuando los nombres son largos
accidentes_por_localidad.sort_values(ascending=True).plot(
    kind='barh',
    figsize=(10, 8),
    color='steelblue',
    title='Accidentes por localidad en Bogotá (2016)'
)
plt.xlabel('Número de accidentes')
plt.show()

> **Tip:** Usamos `barh` (barras horizontales) en lugar de `bar` cuando las etiquetas de las categorías son largas. Es más legible.

**Pregunta:** ¿Qué tipo de accidentes son los más frecuentes?

In [ ]:
base_de_datos['ClaseNombre'].value_counts().plot(
    kind='bar',
    figsize=(10, 4),
    color='coral',
    edgecolor='white',
    title='Tipos de accidentes más frecuentes',
    rot=45
)
plt.ylabel('Cantidad')
plt.show()

**Pregunta:** ¿Cómo influyen las condiciones climáticas en los accidentes?

In [ ]:
base_de_datos['TipoTiempo'].value_counts().head(5).plot(
    kind='barh',
    figsize=(8, 4),
    color='slategray',
    title='Accidentes por condición climática'
)
plt.xlabel('Cantidad de accidentes')
plt.show()

---
### 2.4 Gráfico de dispersión: Relación entre dos variables

**Pregunta:** ¿Existe relación entre el número de muertos y heridos en los accidentes?

In [ ]:
base_de_datos.plot(
    x='TotalMuertos', 
    y='TotalHeridos', 
    kind='scatter',
    figsize=(8, 5),
    alpha=0.3,
    title='Relación entre muertos y heridos por accidente',
    xlabel='Número de muertos',
    ylabel='Número de heridos'
)
plt.show()

> **`alpha`** controla la transparencia de los puntos (0 = invisible, 1 = sólido). Es útil cuando hay muchos puntos superpuestos.

---
### 2.5 Boxplot: Comparar distribuciones entre grupos

El boxplot muestra la **mediana**, los **cuartiles** y los **valores atípicos** de una distribución. Es ideal para comparar grupos.

**Pregunta:** ¿Cómo se distribuye el número de heridos según la gravedad del accidente?

In [ ]:
base_de_datos.boxplot(
    column='TotalHeridos', 
    by='GravedadNombre',
    figsize=(8, 5)
)
plt.title('Distribución de heridos por gravedad del accidente')
plt.suptitle('')  # Quita el título automático que genera pandas
plt.ylabel('Total de heridos')
plt.show()

**Pregunta:** ¿Cómo se distribuye el número de heridos según el tipo de accidente?

In [ ]:
base_de_datos.boxplot(
    column='TotalHeridos', 
    by='ClaseNombre',
    figsize=(12, 5),
    rot=45
)
plt.title('Distribución de heridos por tipo de accidente')
plt.suptitle('')
plt.ylabel('Total de heridos')
plt.tight_layout()
plt.show()

---
### 2.6 Gráfico de torta: Proporciones (paso a paso)

**Pregunta:** ¿Qué proporción de accidentes resultan en daños, heridos o muertos?

El gráfico de torta tiene muchas opciones de personalización. Vamos a construirlo paso a paso.

**Paso 1:** Preparamos los datos. Para un gráfico de torta necesitamos una Serie con los valores y un índice con las categorías.

In [ ]:
tipos_accidentes = base_de_datos['GravedadNombre'].value_counts()
tipos_accidentes

**Paso 2:** El gráfico más básico:

In [ ]:
tipos_accidentes.plot(kind='pie', figsize=(6, 6));

Funciona, pero tiene problemas: la etiqueta del eje Y estorba y no se ven los porcentajes.

**Paso 3:** Agregamos los porcentajes con `autopct`:

In [ ]:
tipos_accidentes.plot(
    kind='pie',
    figsize=(6, 6),
    autopct='%1.1f%%'  # Formato: 1 decimal + símbolo %
)
plt.ylabel('')  # Quitamos la etiqueta del eje Y
plt.title('Proporción de accidentes por gravedad')
plt.show()

**Paso 4:** Personalizamos los colores manualmente:

In [ ]:
tipos_accidentes.plot(
    kind='pie',
    figsize=(6, 6),
    autopct='%1.1f%%',
    colors=['#3498db', '#e74c3c', '#2c3e50']  # Lista de colores, uno por categoría
)
plt.ylabel('')
plt.title('Proporción de accidentes por gravedad')
plt.show()

**Paso 5:** Separamos una porción con `explode` para resaltarla:

In [ ]:
# explode recibe una tupla con la distancia de separación de cada porción
# 0 = sin separación, 0.1 = separación leve
tipos_accidentes.plot(
    kind='pie',
    figsize=(6, 6),
    autopct='%1.1f%%',
    colors=['#3498db', '#e74c3c', '#2c3e50'],
    explode=(0, 0, 0.15)  # Separamos la tercera porción (Con Muertos)
)
plt.ylabel('')
plt.title('Proporción de accidentes por gravedad')
plt.show()

**Paso 6:** Movemos la leyenda fuera del gráfico y ajustamos etiquetas:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

tipos_accidentes.plot(
    kind='pie',
    ax=ax,
    autopct='%1.1f%%',
    colors=['#3498db', '#e74c3c', '#2c3e50'],
    explode=(0, 0, 0.15),
    startangle=90,       # Ángulo de inicio
    pctdistance=0.75,    # Distancia del porcentaje al centro
    labeldistance=1.1    # Distancia de las etiquetas al centro
)
ax.set_ylabel('')
ax.set_title('Proporción de accidentes por gravedad')
ax.legend(loc='lower right')
plt.show()

**Paso 7 (sin etiquetas, solo leyenda):** A veces las etiquetas se superponen. Podemos quitarlas y dejar solo la leyenda:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

tipos_accidentes.plot(
    kind='pie',
    ax=ax,
    autopct='%1.1f%%',
    colors=['#3498db', '#e74c3c', '#2c3e50'],
    explode=(0, 0, 0.15),
    startangle=90,
    labels=['', '', '']  # Quitamos las etiquetas del gráfico
)
ax.set_ylabel('')
ax.set_title('Proporción de accidentes por gravedad')
ax.legend(tipos_accidentes.index, loc='lower right', title='Gravedad')
plt.show()

**Otro ejemplo:** Proporción de accidentes por tipo (`ClaseNombre`):

In [ ]:
tipos_clase = base_de_datos['ClaseNombre'].value_counts()

fig, ax = plt.subplots(figsize=(7, 6))
tipos_clase.plot(
    kind='pie',
    ax=ax,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8
)
ax.set_ylabel('')
ax.set_title('Proporción por tipo de accidente')
plt.show()

> Cuando hay muchas categorías con proporciones similares, el gráfico de torta se vuelve difícil de leer. En esos casos, un gráfico de barras comunica mejor.

---
### 2.7 Personalización: Colores

Hay varias formas de especificar colores en Matplotlib:

**1. Colores por nombre:** Matplotlib tiene colores predefinidos como `'darkred'`, `'steelblue'`, `'forestgreen'`, etc.

<img src="./img/colors.png" width="700">

**2. Códigos HEX:** Permiten especificar cualquier color con formato `'#RRGGBB'`

<img src="./img/HEX.png" width="400">

Pueden buscar colores HEX en Google buscando "HEX color picker".

In [ ]:
# Ejemplo: mismo gráfico con diferentes colores
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

datos = accidentes_diarios.set_index('Fecha')['Número de accidentes']

datos.plot(ax=axes[0], color='forestgreen', title='Color por nombre')
datos.plot(ax=axes[1], color='#e74c3c', title='Color HEX: #e74c3c')
datos.plot(ax=axes[2], color='#9b59b6', title='Color HEX: #9b59b6')

plt.tight_layout()
plt.show()

---
## Bloque 3: Seaborn

**Seaborn** es una librería construida sobre Matplotlib que facilita la creación de gráficos estadísticos. Sus principales ventajas:

1. **Gráficos más bonitos** con menos código
2. **El parámetro `hue`**: permite agregar una dimensión categórica a cualquier gráfico con una sola línea
3. **Mejor integración con DataFrames** de Pandas

In [ ]:
import seaborn as sns
# %pip install seaborn  # Si no está instalado

### Paletas de colores en Seaborn

Seaborn tiene paletas predefinidas que facilitan elegir conjuntos de colores armónicos.

<img src="./img/palette.png" width="700">

In [ ]:
# Podemos explorar paletas con sns.color_palette()
sns.color_palette('Spectral')

In [ ]:
sns.color_palette('Paired')

In [ ]:
sns.color_palette('Blues')

In [ ]:
# También podemos ver la paleta como un colormap continuo
sns.color_palette('viridis', as_cmap=True)

In [ ]:
# Las paletas con _r al final son la versión invertida
sns.color_palette('RdYlGn')

In [ ]:
sns.color_palette('RdYlGn_r')

Podemos usar estas paletas para colorear nuestros gráficos. Por ejemplo, en el gráfico de torta:

In [ ]:
# Usamos una paleta de Seaborn para el pie chart
colores_spectral = sns.color_palette('Spectral', n_colors=3)

tipos_accidentes.plot(
    kind='pie',
    figsize=(6, 6),
    autopct='%1.1f%%',
    colors=colores_spectral
)
plt.ylabel('')
plt.title('Proporción por gravedad (paleta Spectral)')
plt.show()

---
### 3.1 Histograma con Seaborn

Seaborn agrega la opción de **KDE** (Kernel Density Estimation): una curva suavizada que estima la distribución.

In [ ]:
plt.figure(figsize=(12, 4))
sns.histplot(base_de_datos['Hora'], bins=24, kde=True, color='navy')
plt.title('Distribución de accidentes por hora (con KDE)')
plt.xlabel('Hora del día')
plt.show()

---
### 3.2 Boxplot con Seaborn

En Seaborn, el boxplot es mucho más simple y con mejor estética por defecto.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='GravedadNombre', y='TotalHeridos', data=base_de_datos)
plt.title('Distribución de heridos por gravedad')
plt.xlabel('Gravedad')
plt.ylabel('Total de heridos')
plt.show()

In [ ]:
# Boxplot por tipo de diseño de vía
plt.figure(figsize=(14, 5))
sns.boxplot(x='TipoDiseño', y='TotalHeridos', data=base_de_datos)
plt.title('Distribución de heridos por tipo de diseño de vía')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
### 3.3 El poder de `hue`: agregar una dimensión categórica

El parámetro `hue` es la gran ventaja de Seaborn. Permite **colorear los datos según una categoría** con una sola línea de código.

Veamos la diferencia con un ejemplo:

In [ ]:
# Datos de ejemplo simples
np.random.seed(0)
df_ejemplo = pd.DataFrame({
    'x': np.random.rand(100),
    'y': np.random.rand(100),
    'categoría': np.random.choice(['A', 'B', 'C'], size=100)
})
df_ejemplo.head()

**Con Seaborn** (una línea de código):

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_ejemplo, x='x', y='y', hue='categoría')
plt.title("Scatterplot con 'hue' en Seaborn")
plt.show()

**Con Matplotlib puro** (necesitas un loop manual, colores explícitos, y construir la leyenda):

In [ ]:
categorias = df_ejemplo['categoría'].unique()
colores = {'A': 'blue', 'B': 'orange', 'C': 'green'}

plt.figure(figsize=(8, 5))
for cat in categorias:
    subset = df_ejemplo[df_ejemplo['categoría'] == cat]
    plt.scatter(subset['x'], subset['y'], label=cat, color=colores[cat], alpha=0.6)

plt.legend()
plt.title('Scatterplot manual en Matplotlib')
plt.show()

El resultado es similar, pero Seaborn lo hace en **1 línea** vs **6 líneas** de Matplotlib.

El parámetro `hue` funciona en casi todos los gráficos de Seaborn: `boxplot`, `histplot`, `scatterplot`, `barplot`, etc.

---
### 3.4 Usando `hue` con el dataset de accidentes

**Pregunta:** ¿La distribución horaria de accidentes cambia según la gravedad?

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(
    data=base_de_datos, 
    x='Hora', 
    hue='GravedadNombre', 
    bins=24,
    multiple='dodge'  # Barras lado a lado (en vez de apiladas)
)
plt.title('Distribución horaria de accidentes por gravedad')
plt.xlabel('Hora del día')
plt.show()

**Pregunta:** ¿Cómo varía el número de heridos según el tipo de accidente y la gravedad?

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(
    data=base_de_datos, 
    x='ClaseNombre', 
    y='TotalHeridos', 
    hue='GravedadNombre'
)
plt.title('Heridos por tipo de accidente y gravedad')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
### 3.5 Múltiples gráficos con `plt.subplots()`

A veces queremos comparar gráficos lado a lado o uno debajo del otro.

In [ ]:
fig, ejes = plt.subplots(3, figsize=(10, 15))

gravedades = ['Con Heridos', 'Solo Daños', 'Con Muertos']
colores_gravedad = sns.color_palette('Set2', n_colors=3)

for i, gravedad in enumerate(gravedades):
    datos_filtrados = base_de_datos[base_de_datos['GravedadNombre'] == gravedad]
    ejes[i].hist(
        datos_filtrados['Hora'], 
        bins=24, 
        color=colores_gravedad[i], 
        edgecolor='white'
    )
    ejes[i].set_title(f'Distribución horaria: {gravedad}')
    ejes[i].set_xlabel('Hora del día')
    ejes[i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

**Pregunta:** ¿Cómo se compara la accidentalidad mensual entre tipos de accidente?

In [ ]:
# Ejemplo con subplots lado a lado
fig, ejes = plt.subplots(1, 3, figsize=(18, 4))

top3_clases = base_de_datos['ClaseNombre'].value_counts().head(3).index

for i, clase in enumerate(top3_clases):
    datos_filtrados = base_de_datos[base_de_datos['ClaseNombre'] == clase]
    datos_filtrados.groupby('Mes').size().plot(
        ax=ejes[i],
        kind='bar',
        color=sns.color_palette('Set1')[i],
        title=clase,
        rot=0
    )
    ejes[i].set_xlabel('Mes')
    ejes[i].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()

---
## Bloque 4: Ejercicios

### Ejercicio integrador: Dataset de accidentes

Para cada pregunta, **decide tú qué tipo de gráfico es el más apropiado** y créalo. No hay una única respuesta correcta, pero justifica tu elección.

**1.** ¿En qué meses del año ocurren más accidentes? ¿Se ve alguna tendencia?

**2.** ¿Hay diferencia en la distribución horaria de accidentes entre las localidades con más accidentes (top 5)?

**3.** ¿Qué día de la semana tiene más accidentes? ¿Cambia según la gravedad?

**4.** ¿Qué tipo de diseño de vía (`TipoDiseño`) tiene más accidentes fatales (Con Muertos)?

**5.** ¿Cómo se distribuyen los tipos de accidentes (`ClaseNombre`) según las condiciones climáticas (`TipoTiempo`)?

In [ ]:
# Ejercicio 1


In [ ]:
# Ejercicio 2


In [ ]:
# Ejercicio 3


In [ ]:
# Ejercicio 4


In [ ]:
# Ejercicio 5


---
### Ejercicios con el dataset Iris

El dataset **Iris** es un clásico en ciencia de datos. Contiene medidas de 150 flores de 3 especies diferentes (*setosa*, *versicolor*, *virginica*).

Las variables disponibles son:
- `sepal_length`: Longitud del sépalo
- `sepal_width`: Ancho del sépalo
- `petal_length`: Longitud del pétalo
- `petal_width`: Ancho del pétalo
- `species`: Especie de la flor

In [ ]:
iris = sns.load_dataset('iris')
iris.head()

In [ ]:
iris.describe()

**Ejercicio 1:** Realiza un histograma usando Matplotlib que muestre la distribución de `sepal_length`.

In [ ]:
# Ejercicio 1 - Iris


**Ejercicio 2:** Crea un boxplot usando Seaborn que compare la distribución de `petal_length` entre las 3 especies.

In [ ]:
# Ejercicio 2 - Iris


**Ejercicio 3:** Crea un scatterplot de `sepal_width` vs `sepal_length`, coloreado por especie usando `hue`.

In [ ]:
# Ejercicio 3 - Iris


**Ejercicio 4 (Bonus):** Seaborn tiene una función llamada `pairplot` que grafica todas las combinaciones de variables numéricas. Úsala sobre el dataset Iris con `hue='species'`. ¿Qué puedes concluir sobre qué variables separan mejor las especies?

In [ ]:
# Ejercicio 4 (Bonus) - Iris
